In [ ]:
import networkx as nx
G = nx.read_graphml("GGraphInstances/newGeneratedInstances/27x27Graphs/Center486_G0.graphml")
print(len(G), "nodes", len(G.edges), "edges")

In [ ]:
import os
import json
import time
import networkx as nx
from rl import TerritoryDesignProblem, BVNS

model_path = "rl_model.pt"
results_folder = "Results/MyExperiments"
os.makedirs(results_folder, exist_ok=True)

print("=" * 50)
print("PHASE 1: TRAINING on planar500")
print("=" * 50)

for i in range(10):
    instance_name = f"planar500_G{i}"
    file_path = f"TGraphInstances/{instance_name}.graphml"
    print(f"Training on {instance_name}...")

    G = nx.read_graphml(file_path)
    tdp = TerritoryDesignProblem(
        graph_input=G, delta=0.05, llambda=0.4,
        rcl_parameter=0.2, nr_districts=10
    )
    bvns = BVNS(
        tdp_instance=tdp, shaking_steps=25,
        fail_max=50, nrInitSolutions=50
    )

    bvns.agent.load(model_path)      
    bvns.performBVNS(train_agent=True) 
    bvns.agent.save(model_path)      

print("\nPhase 1 done. Model saved to", model_path)


print("\n" + "=" * 50)
print("PHASE 2: EVALUATING all instances")
print("=" * 50)

for size in [500, 600, 700]:
    for i in range(10):
        instance_name = f"planar{size}_G{i}"
        file_path = f"TGraphInstances/{instance_name}.graphml"
        print(f"\nEvaluating {instance_name}...")

        G = nx.read_graphml(file_path)
        start_time = time.time()

        tdp = TerritoryDesignProblem(
            graph_input=G, delta=0.05, llambda=0.4,
            rcl_parameter=0.2, nr_districts=10
        )
        bvns = BVNS(
            tdp_instance=tdp, shaking_steps=25,
            fail_max=50, nrInitSolutions=50
        )

        bvns.agent.load(model_path)       
        obj_hist, inf_hist, best_solution, timeline = bvns.performBVNS(train_agent=False)  # tắt train

        end_time = time.time()

        result = {
            "Instance": instance_name,
            "Objective": obj_hist[-1],
            "Infeasibility": inf_hist[-1],
            "Time Taken": end_time - start_time,
            "nrDistricts": len(best_solution),
            "delta": tdp.delta,
            "llambda": tdp.llambda,
            "Allocation": {
                str(k): [str(n) for n in nodes]
                for k, nodes in best_solution.items()
            }
        }

        save_path = os.path.join(results_folder, f"{instance_name}.json")
        with open(save_path, "w") as f:
            json.dump(result, f, indent=4)

        print(f"Saved → {save_path}")


In [ ]:
import json
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.cm as cm


# ==============================
# 1. LOAD GRAPH
# ==============================
def load_graph(graph_path):
    G = nx.read_graphml(graph_path)

    # Convert node labels -> int (rất quan trọng)
    G = nx.relabel_nodes(G, lambda x: int(x))

    # Lấy tọa độ node
    pos = {n: (float(data["x"]), float(data["y"])) for n, data in G.nodes(data=True)}

    return G, pos


# ==============================
# 2. LOAD RESULT
# ==============================
def load_result(result_path):
    with open(result_path, "r") as f:
        data = json.load(f)

    allocation_raw = data["Allocation"]

    # Convert string -> int
    districts = {int(k): [int(n) for n in v] for k, v in allocation_raw.items()}

    return districts


# ==============================
# 3. VISUALIZE
# ==============================
def plot_districts(G, districts, pos):
    plt.figure(figsize=(10, 10))

    palette = cm.get_cmap("tab20")

    # --- Draw edges ---
    nx.draw_networkx_edges(G, pos, width=0.2, alpha=0.3)

    # --- Draw nodes by district ---
    for k, nodes in districts.items():
        nx.draw_networkx_nodes(
            G,
            pos,
            nodelist=nodes,
            node_color=[palette(k % 20)],
            node_size=20,
            label=f"D{k}",
        )

        # --- Draw centroid ---
        cx = sum(pos[n][0] for n in nodes) / len(nodes)
        cy = sum(pos[n][1] for n in nodes) / len(nodes)

        plt.scatter(cx, cy, color="black", marker="x", s=60)

    # --- Style ---
    plt.title("DTDP Solution (BVNS / MIP)", fontsize=14)
    plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left", fontsize=8)
    plt.axis("equal")
    plt.axis("off")

    plt.show()


# ==============================
# 4. MAIN
# ==============================

graph_path = "TGraphInstances/planar500_G0.graphml"
result_path = "Results/MyExperiments/planar500_G0.json"


G, pos = load_graph(graph_path)
districts = load_result(result_path)
plot_districts(G, districts, pos)


In [ ]:
import json
import os
import pandas as pd

sizes = [500, 600, 700]
rows = []

for size in sizes:
    for i in range(10):
        name = f"planar{size}_G{i}"
        my_path = f"Results/MyExperiments/{name}.json"
        vns_path = f"Results/VNSExperiments/{name}_vns.json"

        row = {"Instance": name, "Size": size}

        if os.path.exists(my_path):
            d = json.load(open(my_path))
            row["BVNS_RL_Obj"]  = round(d["Objective"], 3)
            row["BVNS_RL_Inf"]  = round(d["Infeasibility"], 4)
            row["BVNS_RL_Time"] = round(d["Time Taken"], 1)
        else:
            row["BVNS_RL_Obj"] = row["BVNS_RL_Inf"] = row["BVNS_RL_Time"] = None

        if os.path.exists(vns_path):
            d = json.load(open(vns_path))
            row["VNS_Obj"]  = round(d["Objective"], 3)
            row["VNS_Inf"]  = round(d["Infeasibility"], 4)
            row["VNS_Time"] = round(d["Time Taken"], 1)
        else:
            row["VNS_Obj"] = row["VNS_Inf"] = row["VNS_Time"] = None

        rows.append(row)

df = pd.DataFrame(rows)

# Cột so sánh
df["Obj_Diff"]  = df["BVNS_RL_Obj"] - df["VNS_Obj"]   # âm = BVNS_RL tốt hơn
df["Time_Diff"] = df["BVNS_RL_Time"] - df["VNS_Time"]  # âm = BVNS_RL nhanh hơn

# ── Bảng chi tiết từng instance ──────────────────────────────────
print("=" * 85)
print(f"{'Instance':<20} {'BVNS-RL Obj':>12} {'VNS Obj':>10} {'Diff':>8} {'BVNS-RL Time':>14} {'VNS Time':>10} {'Diff':>8}")
print("=" * 85)
for _, r in df.iterrows():
    print(f"{r['Instance']:<20} {r['BVNS_RL_Obj']:>12} {r['VNS_Obj']:>10} {r['Obj_Diff']:>+8.3f} {r['BVNS_RL_Time']:>14} {r['VNS_Time']:>10} {r['Time_Diff']:>+8.1f}")

# ── Tổng hợp theo kích thước ─────────────────────────────────────
print("\n" + "=" * 60)
print("TỔNG HỢP THEO KÍCH THƯỚC")
print("=" * 60)
for size in sizes:
    sub = df[df["Size"] == size]
    print(f"\nplanar{size} (n={len(sub)}):")
    print(f"  Objective  — BVNS-RL avg: {sub['BVNS_RL_Obj'].mean():.3f}  |  VNS avg: {sub['VNS_Obj'].mean():.3f}  |  Diff: {sub['Obj_Diff'].mean():+.3f}")
    print(f"  Time (s)   — BVNS-RL avg: {sub['BVNS_RL_Time'].mean():.1f}   |  VNS avg: {sub['VNS_Time'].mean():.1f}   |  Diff: {sub['Time_Diff'].mean():+.1f}")
    print(f"  BVNS-RL wins (Obj): {(sub['Obj_Diff'] < 0).sum()}/{len(sub)} instance")

# ── Tổng hợp toàn bộ ─────────────────────────────────────────────
print("\n" + "=" * 60)
print("TỔNG HỢP TOÀN BỘ (30 instances)")
print("=" * 60)
print(f"  Objective  — BVNS-RL avg: {df['BVNS_RL_Obj'].mean():.3f}  |  VNS avg: {df['VNS_Obj'].mean():.3f}  |  Diff: {df['Obj_Diff'].mean():+.3f}")
print(f"  Time (s)   — BVNS-RL avg: {df['BVNS_RL_Time'].mean():.1f}   |  VNS avg: {df['VNS_Time'].mean():.1f}   |  Diff: {df['Time_Diff'].mean():+.1f}")
print(f"  BVNS-RL wins (Obj): {(df['Obj_Diff'] < 0).sum()}/{len(df)} instance")
print(f"  BVNS-RL faster:     {(df['Time_Diff'] < 0).sum()}/{len(df)} instance")
